In [66]:
import pandas as pd
import os


In [89]:

import psycopg2
from psycopg2.extras import execute_values

In [67]:
os.getcwd()

'/Users/sajinak/Retail ETL Pipeline Assessment'

In [68]:
df = pd.read_csv("store_sales.csv")

In [69]:
df.describe()

,order_line_item,quantity,unit_price,store_discount,payment_gateway_discount,tax_amount,amount_paid
count,25000.000000,25000.000000,25000.000000,25000.000000,25000.000000,25000.000000,25000.000000
mean,1.567640,3.021280,210.321865,31.461608,0.730749,48.095892,649.262059
std,0.727025,1.411363,303.936768,72.333802,3.173807,79.986931,1079.811572
min,1.000000,1.000000,5.000000,0.000000,-5.000000,0.000000,-4.960000
25%,1.000000,2.000000,25.240000,0.000000,0.000000,5.220000,70.452500
50%,1.000000,3.000000,78.815000,4.750000,0.000000,14.760000,199.230000
75%,2.000000,4.000000,186.187500,23.732500,0.000000,45.232500,610.775000
max,3.000000,5.000000,1199.760000,598.240000,19.980000,477.960000,6452.510000


In [70]:
df.head()

,order_id,order_line_item,order_date,customer_email,customer_city,store_code,product_sku,category,quantity,unit_price,store_discount,payment_gateway_discount,tax_amount,amount_paid,transaction_type
0,ORD-10001,1,2021-11-16,customer_1@email.com,Seattle,STORE-SEA04,SKU-H301,Home,2,141.01,0.00,0.0,22.56,304.58,SALE
1,ORD-10002,1,2025-06-30,customer_2@email.com,Miami,STORE-MIA05,SKU-H301,Home,1,53.49,2.67,0.0,4.07,54.89,SALE
2,ORD-10003,1,2023-04-18,customer_3@email.com,Chicago,STORE-CHI02,SKU-A202,Apparel,3,106.39,31.92,0.0,22.98,310.23,SALE
3,ORD-10004,1,2022-10-17,customer_4@email.com,Austin,STORE-AUS07,SKU-E101,Electronics,2,659.74,0.00,0.0,105.56,1425.04,SALE
4,ORD-10005,1,2025-03-31,customer_5@email.com,Chicago,STORE-CHI02,SKU-H301,Home,4,131.85,0.00,0.0,42.19,569.59,SALE


In [71]:
df.tail()

,order_id,order_line_item,order_date,customer_email,customer_city,store_code,product_sku,category,quantity,unit_price,store_discount,payment_gateway_discount,tax_amount,amount_paid,transaction_type
24995,ORD-24156,1,2025-08-06,customer_4156@email.com,Chicago,STORE-CHI02,SKU-E102,Electronics,4,366.67,73.33,0.0,111.47,1504.82,SALE
24996,ORD-24156,2,2025-08-06,customer_4156@email.com,Chicago,STORE-CHI02,SKU-A201,Apparel,1,31.19,1.56,0.0,2.37,32.00,SALE
24997,ORD-24157,1,2024-09-21,customer_4157@email.com,New York,STORE-NY01,SKU-F404,Food,1,27.21,0.00,0.0,2.18,19.39,SALE
24998,ORD-24157,2,2024-09-21,customer_4157@email.com,New York,STORE-NY01,SKU-F404,Food,5,27.75,6.94,0.0,10.54,142.35,SALE
24999,ORD-24157,3,2024-09-21,customer_4157@email.com,New York,STORE-NY01,SKU-F404,Food,4,19.78,7.91,0.0,5.70,76.91,SALE


In [72]:
len(df)

25000

In [73]:
# TODO 1

df['order_date'] = pd.to_datetime(df['order_date'],errors = 'coerce')

In [74]:
invalid_dates_count = df['order_date'].isna().sum()
print('invalid dates found:', invalid_dates_count)

invalid dates found: 0


In [75]:
if invalid_dates_count > 0:
    df = df.dropna(subset = ['order_date']).copy()

In [76]:
df['order_date'] = df['order_date'].dt.date
df['order_date'].head(4)

0    2021-11-16
1    2025-06-30
2    2023-04-18
3    2022-10-17
Name: order_date, dtype: object

In [77]:
# TODO 2

missing_emails = df['customer_email'].isna().sum()

In [78]:
print("No of Missing emails - ",missing_emails)

No of Missing emails -  0


In [79]:
# here im doing some string operations to make sure the columns data look good 
df['customer_email'] = (
    df['customer_email']
    .astype(str)
    .str.strip()
    .str.lower()
    .replace({'nan': None,'':  None,'none': None})
    .fillna('missing_customer@email.com')
    
)

In [80]:
df['customer_email'].head(4)

0    customer_1@email.com
1    customer_2@email.com
2    customer_3@email.com
3    customer_4@email.com
Name: customer_email, dtype: str

In [81]:
# TODO 3 here im filtering the corrupt records

df['quantity'] = pd.to_numeric(df['quantity'],errors = 'coerce').fillna(0).astype(int)


In [82]:
invalid_quantity = df['quantity'] <= 0
print('invalid quantities found - ',invalid_quantity.sum())

if invalid_quantity.sum() > 0:
    df = df[~invalid_quantity].copy()

invalid quantities found -  0


In [83]:
monetary_columns = [
    'unit_price',
    'store_discount',
    'payment_gateway_discount',
    'tax_amount',
    'amount_paid'
]

for col in monetary_columns:
    df[col] = pd.to_numeric(df[col],errors = 'coerce').fillna(0.0).round(2)


In [84]:
# TODO 4 here i count and delete the duplicated values

dup_count = df.duplicated(subset = ['order_id','order_line_item'], keep ='first').sum()
print("duplicated count - ",dup_count)

duplicated count -  345


In [85]:
df = df.drop_duplicates(subset = ['order_id','order_line_item'],keep = 'first').copy()
print('After droping duplicates - ',df)

After droping duplicates -          order_id  order_line_item  order_date           customer_email  \
0      ORD-10001                1  2021-11-16     customer_1@email.com   
1      ORD-10002                1  2025-06-30     customer_2@email.com   
2      ORD-10003                1  2023-04-18     customer_3@email.com   
3      ORD-10004                1  2022-10-17     customer_4@email.com   
4      ORD-10005                1  2025-03-31     customer_5@email.com   
...          ...              ...         ...                      ...   
24995  ORD-24156                1  2025-08-06  customer_4156@email.com   
24996  ORD-24156                2  2025-08-06  customer_4156@email.com   
24997  ORD-24157                1  2024-09-21  customer_4157@email.com   
24998  ORD-24157                2  2024-09-21  customer_4157@email.com   
24999  ORD-24157                3  2024-09-21  customer_4157@email.com   

      customer_city   store_code product_sku     category  quantity  \
0           

In [ ]:
conn = psycopg2.connect(
    host = '',
    database = '',
    port = ,
    user = ''
)
cursor = conn.cursor()

In [111]:
cursor.execute("""
    CREATE TABLE IF NOT EXISTS fact_retail_sales (
        order_id VARCHAR(50),
        order_line_item INT,
        order_date DATE,
        customer_email VARCHAR(100),
        customer_city VARCHAR(100),
        store_code VARCHAR(50),
        product_sku VARCHAR(50),
        category VARCHAR(50),
        quantity INT,
        unit_price NUMERIC(10,2),
        store_discount NUMERIC(10,2),
        payment_gateway_discount NUMERIC(10,2),
        tax_amount NUMERIC(10,2),
        amount_paid NUMERIC(10,2),
        transaction_type VARCHAR(20),
        loaded_at TIMESTAMP DEFAULT CURRENT_TIMESTAMP,
        PRIMARY KEY (order_id,order_line_item)
        
    )

""")
conn.commit()
print("connected to postgreSQL and table created successfully :)")

connected to postgreSQL and table created successfully :)


In [118]:
columns = [
    'order_id',
    'order_line_item',
    'order_date',
    'customer_email',
    'customer_city',
    'store_code',
    'product_sku',
    'category',
    'quantity',
    'unit_price',
    'store_discount',
    'payment_gateway_discount',
    'tax_amount',
    'amount_paid',
    'transaction_type',
    

]

records = [tuple(row) for row in df[columns].itertuples(index=False)]

upsert_query = f"""
    
    INSERT INTO fact_retail_sales ({', '.join(columns)})
    VALUES %s
    ON CONFLICT(order_id,order_line_item) DO UPDATE SET
        order_date = EXCLUDED.order_date,
        customer_email = EXCLUDED.customer_email,
        customer_city = EXCLUDED.customer_city,
        store_code = EXCLUDED.store_code,
        product_sku = EXCLUDED.product_sku,
        category = EXCLUDED.category,
        quantity = EXCLUDED.quantity,
        unit_price = EXCLUDED.unit_price,
        store_discount = EXCLUDED.store_discount,
        payment_gateway_discount = EXCLUDED.payment_gateway_discount,
        tax_amount = EXCLUDED.tax_amount,
        amount_paid = EXCLUDED.amount_paid,
        transaction_type = EXCLUDED.transaction_type,
        loaded_at = CURRENT_TIMESTAMP;
"""

print(f"loading {len(records):,}records into postgreSQL :)")
execute_values(cursor,upsert_query,records,page_size = 50000)
conn.commit()
print("data loaded successfully")

loading 24,655records into postgreSQL :)
data loaded successfully


In [140]:
# lets find the sales,return and revenue

sales_revenue = df.loc[df['transaction_type'] == 'SALE','amount_paid'].sum()
returns_deductions = df.loc[df['transaction_type'] == 'RETURN','amount_paid'].sum()
net_revenue = sales_revenue - returns_deductions

print(f'Sales revenue - {sales_revenue:,.2f}')
print(f'Returns deductions - {returns_deductions}')
print(f'Net Revenue - {net_revenue}')

Sales revenue - 15,292,263.27
Returns deductions - 754244.31
Net Revenue - 14538018.959999999
